# Worked Example: Time-Resolved Regression vs RPE

## Goal
Regress feedback-locked beta power against reward prediction error.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from pathlib import Path
from LFPAnalysis import statistics_utils

np.random.seed(42)
beh = pd.read_csv(Path('../../data/sample_beh.csv'))
epochs = mne.read_epochs(Path('../../data/sample_feedback_start-epo.fif'), preload=True, verbose=False)
epochs.metadata = beh[['reward', 'rpe']]
chan = 'racas1-racas2'
ep = epochs.copy().pick([chan]).filter(13, 30, verbose=False)
times = ep.times
betas, z_betas = [], []
for t_idx in range(0, len(times), max(1, len(times) // 20)):
    power = ep.get_data()[:, 0, t_idx] ** 2
    df = pd.DataFrame({'power': power, 'rpe': epochs.metadata['rpe'].values})
    res = statistics_utils.permutation_regression_zscore(df, 'power ~ rpe', n_permutations=100)
    betas.append(res.loc[res.predictor == 'rpe', 'raw_beta'].values[0])
    z_betas.append(res.loc[res.predictor == 'rpe', 'z_beta'].values[0])
time_sub = times[::max(1, len(times) // 20)]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(time_sub, z_betas, 'o-')
ax.axvline(0, color='k', ls='--', lw=0.8)
ax.axhline(0, color='0.5', ls='-', lw=0.5)
ax.set(xlabel='Time (s)', ylabel='z-beta (RPE)', title=f'{chan} feedback-locked')
fig.tight_layout()
plt.show()

## Next step

Chapter 11 covers advanced utility interoperability.